## Need to add the inital dependecies of the local ollama embedding model


In [17]:
import os

from dotenv import load_dotenv
from ollama import AsyncClient, Client
from sklearn.metrics.pairwise import (
    cosine_similarity,
    euclidean_distances)

load_dotenv()

api_key = os.environ["OLLAMA_API_KEY"]
model = os.getenv("OLLAMA_MODEL", "kimi-k2.7-code:cloud")

client = AsyncClient(
    host="https://ollama.com",
    headers={"Authorization": f"Bearer {api_key}"}
)

# Local client for embeddings (no API key needed)
local_client = Client(host="http://localhost:11434")

EMBEDDING_MODEL = "mxbai-embed-large:latest"

print(f"Chat model: {model}")
print(f"Embedding model: {EMBEDDING_MODEL}")



Chat model: kimi-k2.7-code:cloud
Embedding model: mxbai-embed-large:latest


## Adding some dummy data to the document File.

In [18]:
documents = [
    {"id": 1,  "text": "Spring Boot simplifies building production-ready Java backend applications.",                                               "topic": "backend",           "level": "beginner"},
    {"id": 2,  "text": "Kubernetes orchestrates containers and provides scheduling, service discovery, and self-healing.",                          "topic": "devops",            "level": "intermediate"},
    {"id": 3,  "text": "Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.",                                  "topic": "databases",         "level": "beginner"},
    {"id": 4,  "text": "Kafka is a distributed event streaming platform used to build high-throughput asynchronous systems.",                      "topic": "distributed-systems","level": "intermediate"},
    {"id": 5,  "text": "PostgreSQL is a relational database with strong SQL support, transactions, indexing, and extensibility.",                  "topic": "databases",         "level": "beginner"},
    {"id": 6,  "text": "Vector databases store embeddings and make similarity search over high-dimensional vectors efficient.",                    "topic": "ai",                "level": "beginner"},
    {"id": 7,  "text": "RAG retrieves relevant external context and gives that context to a language model before generating an answer.",          "topic": "ai",                "level": "beginner"},
    {"id": 8,  "text": "An embedding model converts text into a dense numerical vector that captures semantic relationships.",                    "topic": "ai",                "level": "beginner"},
    {"id": 9,  "text": "Cosine similarity compares the angle between vectors and is commonly used for semantic text retrieval.",                  "topic": "ai",                "level": "intermediate"},
    {"id": 10, "text": "HNSW builds a navigable graph so approximate nearest-neighbor search can avoid scanning every vector.",                   "topic": "vector-search",     "level": "advanced"},
    {"id": 11, "text": "Exact nearest-neighbor search compares the query against every candidate vector to obtain the true top results.",         "topic": "vector-search",     "level": "intermediate"},
    {"id": 12, "text": "Quantization compresses vector representations so a vector database can reduce memory usage and improve search efficiency.","topic": "vector-search",     "level": "advanced"},
    {"id": 13, "text": "Payload indexes allow a vector database to combine semantic similarity with structured filters.",                         "topic": "vector-search",     "level": "advanced"},
    {"id": 14, "text": "Python is widely used for machine learning, data processing, and building AI applications.",                              "topic": "programming",       "level": "beginner"},
    {"id": 15, "text": "Async programming can improve throughput for applications that spend significant time waiting on network I/O.",           "topic": "programming",       "level": "intermediate"},
    {"id": 16, "text": "API gateways can provide routing, authentication, rate limiting, and observability for microservices.",                   "topic": "distributed-systems","level": "intermediate"},
    {"id": 17, "text": "Microservices split an application into independently deployable services that communicate over APIs or events.",         "topic": "distributed-systems","level": "intermediate"},
    {"id": 18, "text": "Observability combines logs, metrics, and traces to understand the behavior of distributed applications.",               "topic": "devops",            "level": "intermediate"},
    {"id": 19, "text": "A cache can reduce database load and latency when frequently requested data can be reused safely.",                       "topic": "databases",         "level": "beginner"},
    {"id": 20, "text": "Reranking can improve retrieval quality by applying a more expensive relevance model to an initial candidate set.",       "topic": "ai",                "level": "advanced"},
]

texts = [d["text"] for d in documents]
print(f"{len(documents)} documents")
print(texts[0])

20 documents
Spring Boot simplifies building production-ready Java backend applications.


In [19]:
# So we will do the text to vector embeddings

# def embed_documents(texts: list[str]) -> list[list[float]]:
#     resp = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
#     return [d.embedding for d in resp.data]

# Using the local sync client for embeddings (no async/await needed)
def embed_documents(texts: list[str]) -> list[list[float]]:
    resp = local_client.embed(model=EMBEDDING_MODEL, input=texts)
    return resp.embeddings

doc_embeddings = embed_documents(texts)

print("Number of embeddings:", len(doc_embeddings))
print("Dimensions per vector:", len(doc_embeddings[0]))
print("First 5 values:", doc_embeddings[0])


Number of embeddings: 20
Dimensions per vector: 1024
First 5 values: [0.010558648, 0.0357524, -0.037935562, -0.0623743, -0.044356383, 0.019755172, 0.015065747, -0.0055857343, 0.063076116, 0.039150048, 0.026008938, -0.0063455836, 0.048274025, -0.06250334, -0.02642751, 0.023034759, -0.039008655, -0.011530516, -0.047939382, 0.004530117, 0.029767847, 0.006688931, -0.034968887, -0.046582226, -0.018094702, 0.041532904, 0.010125734, -0.013421465, 0.08043923, 0.02418443, -0.04714344, 0.011580555, 0.028701201, -0.04403196, 0.005668267, -0.008031847, 0.012231009, -0.023312196, 0.0008613483, -0.01608171, 0.053619064, -0.018463008, 0.033808976, -0.06624417, -0.09129945, 0.00865713, 0.008503196, -0.037539102, 0.042195633, 0.02075159, -0.020594109, 0.019959943, -0.00743966, 0.002789203, -0.00637892, 0.014871602, -0.012882868, -0.017905597, -0.0693608, 0.034225322, 0.008731898, 0.044939723, 0.018539896, -0.091692865, 0.013935407, 0.056257825, 0.017051293, -0.0071323495, 0.009739232, -0.022764077, -0.

In [20]:
query = "Which programing language is commonly used for AI applications?"

query_embedding = embed_documents([query])  # returns the list with one vector

print("Query : ", query)
print("Dimensions : ", len(query_embedding[0]))
print("Collection must use size = ", len(doc_embeddings[0]))


Query :  Which programing language is commonly used for AI applications?
Dimensions :  1024
Collection must use size =  1024


## When to use what

| Metric | Intuition | Better match |
|---|---|---|
| Cosine | Angle between vectors | Higher |
| Euclidean / L2 | Straight-line distance | Lower |
| Dot product | Alignment + magnitude | Higher |

In [21]:
cos = cosine_similarity(query_embedding, doc_embeddings)[0]
l2  = euclidean_distances(query_embedding, doc_embeddings)[0]

print(f"{'Cos':>6}  {'L2':>5}  Document")
print("-" * 70)
for doc, c, d in zip(documents, cos, l2):
    print(f"{c:6.3f}  {d:6.3f}  {doc}")

   Cos     L2  Document
----------------------------------------------------------------------
 0.452   1.047  {'id': 1, 'text': 'Spring Boot simplifies building production-ready Java backend applications.', 'topic': 'backend', 'level': 'beginner'}
 0.402   1.093  {'id': 2, 'text': 'Kubernetes orchestrates containers and provides scheduling, service discovery, and self-healing.', 'topic': 'devops', 'level': 'intermediate'}
 0.461   1.038  {'id': 3, 'text': 'Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.', 'topic': 'databases', 'level': 'beginner'}
 0.425   1.072  {'id': 4, 'text': 'Kafka is a distributed event streaming platform used to build high-throughput asynchronous systems.', 'topic': 'distributed-systems', 'level': 'intermediate'}
 0.417   1.080  {'id': 5, 'text': 'PostgreSQL is a relational database with strong SQL support, transactions, indexing, and extensibility.', 'topic': 'databases', 'level': 'beginner'}
 0.455   1.044  {'id': 6, '

## TO Get the Top K values. As in best search values.

In [22]:
def retrieve(query_emb, doc_embs, docs, top_k=2):
    scores = cosine_similarity(query_emb, doc_embs)[0]

    # pair each document with its score, sort highest first, take top_k
    scored = list(zip(docs, scores))
    # print(scored)
    scored.sort(key=lambda x: x[1], reverse=True)

    return [{"document": doc, "score": float(score)} for doc, score in scored[:top_k]]


print("Top-K results:\n")
for hit in retrieve(query_embedding, doc_embeddings, documents, top_k=2):
    print(f"{hit['score']:.3f}  {hit['document']}")

Top-K results:

0.752  {'id': 14, 'text': 'Python is widely used for machine learning, data processing, and building AI applications.', 'topic': 'programming', 'level': 'beginner'}
0.503  {'id': 7, 'text': 'RAG retrieves relevant external context and gives that context to a language model before generating an answer.', 'topic': 'ai', 'level': 'beginner'}


---
## Quadrant vector database

In [24]:
from qdrant_client import QdrantClient, models

#qdrant = QdrantClient(":memory:")

qdrant = QdrantClient(url="http://localhost:6333")

In [25]:
COLLECTION = "rag_fundamentals"

if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(
        size=len(doc_embeddings[0]),   # always read from the model, never hard-code
        distance=models.Distance.COSINE,
    ),
)

print("Created:", COLLECTION, "| size:", len(doc_embeddings[0]))

Created: rag_fundamentals | size: 1024


## Notes 

1. Inserting and updating the points if needed. 
2. Array of points
3. Zip --> Basically if you want to return a tupple of both the object. 

In [26]:
points = [
    models.PointStruct(
        id=doc["id"],
        vector=vec,
        payload={"text": doc["text"], "topic": doc["topic"], "level": doc["level"]},
    )
    for doc, vec in zip(documents, doc_embeddings)
]

qdrant.upsert(collection_name=COLLECTION, points=points, wait=True)
print(f"Inserted {len(points)} points.")

Inserted 20 points.


## Quadrant search

In [27]:
results = qdrant.query_points(
    collection_name=COLLECTION,
    query=query_embedding[0],    # a single vector, not a list
    limit=5,
    with_payload=True,
).points

for p in results:
    print(p)

id=14 version=1 score=0.75199157 payload={'text': 'Python is widely used for machine learning, data processing, and building AI applications.', 'topic': 'programming', 'level': 'beginner'} vector=None shard_key=None order_value=None
id=7 version=1 score=0.50305665 payload={'text': 'RAG retrieves relevant external context and gives that context to a language model before generating an answer.', 'topic': 'ai', 'level': 'beginner'} vector=None shard_key=None order_value=None
id=15 version=1 score=0.47958422 payload={'text': 'Async programming can improve throughput for applications that spend significant time waiting on network I/O.', 'topic': 'programming', 'level': 'intermediate'} vector=None shard_key=None order_value=None
id=8 version=1 score=0.46357697 payload={'text': 'An embedding model converts text into a dense numerical vector that captures semantic relationships.', 'topic': 'ai', 'level': 'beginner'} vector=None shard_key=None order_value=None
id=3 version=1 score=0.4610573 pay

In [29]:
compressedClient = qdrant.create_collection(
    collection_name="Compressed",
    vectors_config=models.VectorParams(size=len(doc_embeddings[0]), distance=models.Distance.COSINE),
    quantization_config=models.ScalarQuantization(
        scalar=models.ScalarQuantizationConfig(
            type=models.ScalarType.INT8,   # float32 -> 8-bit integers
            always_ram=True,
        )
    ),
)

In [30]:
#Trying to insert the points in the new compressed client based on the new bits size as Int8

qdrant.upsert(collection_name="Compressed", points=points, wait=True)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [32]:
results = qdrant.query_points(
    collection_name="Compressed",
    query=query_embedding[0],
    limit=5,
    with_payload=True,
    with_vectors=True,
).points

print("First 5 values of vector:", results[0].vector[:5])

First 5 values of vector: [0.034554422, 0.023894385, -0.034249544, -0.03598305, 0.004030858]
